
# Step 01 — Loading libraries and the 10x data

**Purpose.** Read the eight Cell Ranger filtered matrices into one AnnData object
with clean, correctly ordered metadata, and establish two facts the rest of the
project depends on:

1. whether the **EGFP transgene is present as a feature** in the count matrices;
2. how many cells and genes each sample contributes *before* any filtering.

**Biological context.** Each sample is one 10x run from a pool of dissected adult
zebrafish retinas: two uninjured controls and two replicates each at 3, 7 and 10
days post-MNU. Condition and sequencing run are fully confounded by design — a
limitation to carry through to the Discussion, not something to correct away.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable regardless of where Jupyter was launched.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts").exists(), (
    f"Cannot locate the repository root from {Path.cwd()}. "
    "Launch JupyterLab from the repository root."
)
sys.path.insert(0, str(REPO_ROOT))

from scripts import config as cfg
from scripts import io_utils, qc, preprocessing, clustering, annotation, egfp_analysis, plotting

cfg.ensure_directories()
plotting.set_style()
SEED = io_utils.set_seeds()
print(f"Repository root: {REPO_ROOT}")
print(f"Random seed: {SEED}")

Repository root: D:\retina\project1_zebrafish_retina
Random seed: 0



## 1. Discover and validate the sample directories

Paths are discovered relative to the repository root. Nothing is hard-coded, so
the notebook runs unchanged on another machine.

In [2]:
sample_dirs = io_utils.discover_samples()
expected = io_utils.report_expected_samples(sample_dirs)
display(expected)

missing = expected.loc[expected["expected"] & ~expected["found"], "sample"].tolist()
if missing:
    print(f"WARNING: expected sample(s) not found: {missing}")
    print("The analysis can proceed, but every proportion and time-course result "
          "must state which samples were unavailable.")

,sample,expected,found
0,ctrl1,True,True
1,ctrl2,True,True
2,3dp1,True,True
3,3dp2,True,True
4,7dp1,True,True
5,7dp2,True,True
6,10dp1,True,True
7,10dp2,True,True


In [3]:
# Confirm each folder holds all three Cell Ranger files before reading anything.
for sample_dir in sample_dirs:
    io_utils.validate_sample_dir(sample_dir)
print("All sample directories contain barcodes.tsv.gz, features.tsv.gz and matrix.mtx.gz.")

All sample directories contain barcodes.tsv.gz, features.tsv.gz and matrix.mtx.gz.



## 2. Load and concatenate

`load_all_samples` reads each matrix, attaches `sample` / `condition` /
`replicate` / `timepoint` / `batch`, prefixes barcodes with the sample name so
they stay unique, and concatenates with `join="outer"`.

`condition` is made an **ordered categorical** (`ctrl → 3dp → 7dp → 10dp`) at load
time. Left as a string, `"10dp"` sorts before `"3dp"` and every downstream plot
would silently show the time course out of order.

In [5]:
adata, per_sample = io_utils.load_all_samples()
display(per_sample)

Discovered 8 samples: ['10dp1', '10dp2', '3dp1', '3dp2', '7dp1', '7dp2', 'ctrl1', 'ctrl2']
  10dp1     4124 cells x  25433 genes   EGFP feature: EGFP
  10dp2     2517 cells x  25433 genes   EGFP feature: EGFP
  3dp1      2873 cells x  25433 genes   EGFP feature: EGFP
  3dp2      2388 cells x  25433 genes   EGFP feature: EGFP
  7dp1      3444 cells x  25433 genes   EGFP feature: EGFP
  7dp2      1480 cells x  25433 genes   EGFP feature: EGFP
  ctrl1     1155 cells x  25433 genes   EGFP feature: EGFP
  ctrl2     2116 cells x  25433 genes   EGFP feature: EGFP

Total dataset: 20097 cells x 25433 genes


,sample,condition,replicate,n_cells,n_genes,total_counts,median_counts_per_cell,median_genes_per_cell,egfp_feature
0,10dp1,10dp,1,4124,25433,17932948.0,1232.0,504.0,EGFP
1,10dp2,10dp,2,2517,25433,15987928.0,1275.0,599.0,EGFP
2,3dp1,3dp,1,2873,25433,10324831.0,1095.0,453.0,EGFP
3,3dp2,3dp,2,2388,25433,11730700.0,1308.5,597.5,EGFP
4,7dp1,7dp,1,3444,25433,15630329.0,1012.5,401.0,EGFP
5,7dp2,7dp,2,1480,25433,7171764.0,1314.0,594.0,EGFP
6,ctrl1,ctrl,1,1155,25433,4511528.0,1146.0,574.0,EGFP
7,ctrl2,ctrl,2,2116,25433,4751474.0,975.5,572.5,EGFP


In [6]:
# Assertions on the assumptions the rest of the workflow relies on.
assert adata.n_obs > 0, "No cells loaded."
assert adata.obs_names.is_unique, "Barcodes are not unique."
for col in ("sample", "condition", "replicate", "timepoint", "batch"):
    assert col in adata.obs, f"Missing metadata column: {col}"
assert list(adata.obs["condition"].cat.categories) == cfg.CONDITION_ORDER, (
    "Condition categories are not in biological time order."
)
print("Metadata checks passed.")
print(adata.obs.groupby(["condition", "sample"], observed=True).size().to_string())

Metadata checks passed.
condition  sample
ctrl       ctrl1     1155
           ctrl2     2116
3dp        3dp1      2873
           3dp2      2388
7dp        7dp1      3444
           7dp2      1480
10dp       10dp1     4124
           10dp2     2517



## 3. Is EGFP in the matrix?

The transgene is only countable if it was added to the Cell Ranger reference
before alignment. The feature name is **detected, not assumed** — capitalisation
varies between references (`EGFP`, `eGFP`, `GFP`).

If no EGFP-like feature exists, research question 3 and Figure 4 cannot be
answered from this matrix. The correct response is to report that, not to
substitute a proxy gene.

In [7]:
egfp_name = io_utils.find_egfp_feature(adata)

if egfp_name is not None:
    counts = preprocessing.get_expression(adata, egfp_name, layer="X")
    n_pos = int((counts > 0).sum())
    print(f"Cells with >0 EGFP counts (pre-QC): {n_pos} / {adata.n_obs} "
          f"({100 * n_pos / adata.n_obs:.2f}%)")
    print(f"Max EGFP counts in a single cell: {counts.max():.0f}")
    display(
        adata.obs.assign(egfp_pos=counts > 0)
        .groupby("condition", observed=True)["egfp_pos"]
        .agg(["sum", "size"])
        .assign(percent=lambda d: (100 * d["sum"] / d["size"]).round(2))
    )

EGFP feature found in var_names as: 'EGFP'
Cells with >0 EGFP counts (pre-QC): 554 / 20097 (2.76%)
Max EGFP counts in a single cell: 256


,sum,size,percent
condition,,,
ctrl,16,3271,0.49
3dp,256,5261,4.87
7dp,118,4924,2.40
10dp,164,6641,2.47



### Interpretation

Read the per-condition EGFP table above before going further. The paper reports
EGFP transcripts in ~0.5% of control cells rising to ~5% at 3 dpMNU and settling
near 2.6% at 7 and 10 dpMNU. A **control value near zero and a rise after injury**
is the expected shape. If controls show a high EGFP rate, suspect index hopping or
sample swapping and investigate before interpreting anything else.

In [8]:
# Save the loading checkpoint and the sample-level table.
io_utils.save_table(per_sample, "cell_counts_by_sample_raw.csv")
io_utils.save_checkpoint(adata, "loaded")

Wrote tables\cell_counts_by_sample_raw.csv  (8 rows)
Saved checkpoint 'loaded' -> results\01_loaded.h5ad (52.4 MB)


WindowsPath('D:/retina/project1_zebrafish_retina/results/01_loaded.h5ad')


### What to check before continuing

- [ ] Eight samples discovered; any missing sample is recorded.
- [ ] Cell counts per sample are in a plausible range (thousands, not tens).
- [ ] `condition` categories print as `['ctrl', '3dp', '7dp', '10dp']`.
- [ ] EGFP presence/absence is recorded — this decides whether Figure 4 is possible.
- [ ] `results/01_loaded.h5ad` was written.